In [1]:
!gdown "1cXugsmhdR2kJUEr229dSmvpWMpgHzo0D"

Downloading...
From (original): https://drive.google.com/uc?id=1cXugsmhdR2kJUEr229dSmvpWMpgHzo0D
From (redirected): https://drive.google.com/uc?id=1cXugsmhdR2kJUEr229dSmvpWMpgHzo0D&confirm=t&uuid=3c9296c8-ffaf-4d4e-b878-acf611b81bce
To: /kaggle/working/ip02-dataset.zip
100%|██████████████████████████████████████| 3.16G/3.16G [00:38<00:00, 82.5MB/s]


In [2]:
import os
import shutil
from tqdm import tqdm

# --- 1. CONFIGURATION ---
ZIP_PATH = '/kaggle/working/ip02-dataset.zip' # Update this!
OUTPUT_DIR = '/kaggle/working/ip102_organized'
EXTRACT_DIR = '/kaggle/working/ip102_temp'

# --- 2. FAST EXTRACTION ---
if not os.path.exists(EXTRACT_DIR):
    print("Step 1: Extracting ZIP (Using High-Speed Linux Unzip)...")
    !unzip -q {ZIP_PATH} -d {EXTRACT_DIR}

# --- 3. CREATE FILE INDEX (The Secret to Speed) ---
print("Step 2: Indexing files (this makes moving instant)...")
file_index = {}
for root, dirs, files in os.walk(EXTRACT_DIR):
    for f in files:
        if f.endswith(('.jpg', '.jpeg', '.png')):
            file_index[f] = os.path.join(root, f)

# --- 4. LOAD METADATA ---
def find_meta(name):
    for root, dirs, files in os.walk(EXTRACT_DIR):
        if name in files: return os.path.join(root, name)
    return None

classes_txt = find_meta('classes.txt')
train_txt = find_meta('train.txt')
val_txt = find_meta('val.txt')
test_txt = find_meta('test.txt')

# Build Class Map
classes_map = {}
with open(classes_txt, 'r') as f:
    for line in f:
        p = line.strip().split()
        if p: classes_map[p[0]] = "_".join(p[1:])

# Detect Indexing
with open(train_txt, 'r') as f:
    first_label = f.readline().split()[1]
    is_zero_indexed = (first_label == '0')

# --- 5. ULTRA-FAST ORGANIZATION ---
def fast_organize(split_name, txt_path):
    print(f"Organizing {split_name}...")
    with open(txt_path, 'r') as f:
        lines = f.readlines()

    for line in tqdm(lines):
        parts = line.strip().split()
        if len(parts) < 2: continue
        img_name, label_id = parts[0], parts[1]

        # Handle the 0 vs 1 index automatically
        lookup_id = str(int(label_id) + 1) if is_zero_indexed else label_id

        if lookup_id in classes_map and img_name in file_index:
            class_folder = classes_map[lookup_id]
            dest_dir = os.path.join(OUTPUT_DIR, split_name, class_folder)
            os.makedirs(dest_dir, exist_ok=True)

            # Using os.rename is nearly instantaneous on the same drive
            try:
                os.rename(file_index[img_name], os.path.join(dest_dir, img_name))
            except:
                shutil.copy(file_index[img_name], os.path.join(dest_dir, img_name))

# Run the splits
for s, t in [('train', train_txt), ('val', val_txt), ('test', test_txt)]:
    fast_organize(s, t)

print(f"\nSUCCESS! Data is at {OUTPUT_DIR}")
# Optional: !rm -rf {EXTRACT_DIR}

Step 1: Extracting ZIP (Using High-Speed Linux Unzip)...
Step 2: Indexing files (this makes moving instant)...
Organizing train...


100%|██████████| 45095/45095 [00:01<00:00, 29027.10it/s]


Organizing val...


100%|██████████| 7508/7508 [00:00<00:00, 31365.65it/s]


Organizing test...


100%|██████████| 22619/22619 [00:00<00:00, 28509.43it/s]


SUCCESS! Data is at /kaggle/working/ip102_organized


In [3]:
import torch
import torch.nn as nn
import torchvision.models as models
import torch.nn.functional as F

class TrueRANStyleAttention(nn.Module):
    def __init__(self, in_channels):
        super(TrueRANStyleAttention, self).__init__()
        # 1. Bottom-Up: Zoom out to get global context
        self.downsample = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        # 2. Process features at the lower resolution
        # We shrink the channels by 4 (bottleneck) to save memory, then expand them back
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 4, kernel_size=1, bias=False),
            nn.BatchNorm2d(in_channels // 4),
            nn.ReLU(inplace=True)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(in_channels // 4, in_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(in_channels)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Zoom out
        down = self.downsample(x)
        
        # Build the mask
        mask = self.conv1(down)
        mask = self.conv2(mask)
        
        # 3. Top-Down: Zoom back in to match original feature size
        mask = F.interpolate(mask, size=x.size()[2:], mode='bilinear', align_corners=False)
        mask = self.sigmoid(mask)
        
        # 4. The RAN Innovation: (1 + Mask) * Features
        return x * (1 + mask)



# ... [PASTE YOUR PCSA, PCSABlockWrapper, and get_resnet50_pcsa HERE] ...
class PCSA(nn.Module):
    def __init__(self, in_channels, r=16):
        super().__init__()
        # Channel Branch
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // r),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // r, in_channels),
            nn.Sigmoid() # Keeps weights <= 1.0
        )
        # Spatial Branch
        self.spatial_conv = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid = nn.Sigmoid() # Keeps weights <= 1.0

    def forward(self, x):
        # 1. Get Attention Masks (Sigmoid ensures these are 0.0 to 1.0)
        b, c, _, _ = x.size()

        # Channel Mask
        y_c = x.mean((2, 3))
        channel_mask = self.fc(y_c).view(b, c, 1, 1) # Sigmoid is inside self.fc

        # Spatial Mask
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        y_s = torch.cat([avg_out, max_out], dim=1)
        spatial_mask = self.sigmoid(self.spatial_conv(y_s))

        # 2. THE STABLE FUSION
        # Instead of (x*mask) * (x*mask), we do x * (mask * mask)
        # This applies both attentions to x WITHOUT squaring x.
        combined_mask = channel_mask * spatial_mask
        out = x * combined_mask

        return torch.relu(out)


class PCSABlockWrapper(nn.Module):
    def __init__(self, original_block):
        super().__init__()
        self.block = original_block
        self.pcsa = PCSA(original_block.conv3.out_channels)

    def forward(self, x):
        identity = x
        # Standard ResNet block logic
        out = self.block(x)

        # Apply PCSA as a 'refinement'
        # Adding a residual here ensures that even if PCSA outputs 0,
        # the model doesn't lose the pre-trained features.
        refined = self.pcsa(out)

        return torch.relu(refined + out) # Residual bridge

def get_resnet50_pcsa(num_classes=10):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

    # Inject PCSA into every bottleneck block safely
    for layer_name in ['layer1', 'layer2', 'layer3', 'layer4']:
        layer = getattr(model, layer_name)
        for i in range(len(layer)):
            layer[i] = PCSABlockWrapper(layer[i])

    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


class DeepPCSARANHybrid(nn.Module):
    def __init__(self, num_classes=102):
        super().__init__()
        
        # 1. Build the Deep PCSA Trunk using your custom function
        print("Building Deep PCSA ResNet-50 Trunk...")
        # We pass a dummy num_classes because we are going to delete the final layer anyway
        base_model = get_resnet50_pcsa(num_classes=1000) 
        
        # Strip the final AdaptiveAvgPool2d and Linear layers to make it a feature extractor
        self.trunk = nn.Sequential(*list(base_model.children())[:-2])
        
        # 2. Add the RAN Mask on top of the 2048-channel output
        print("Attaching RAN Module...")
        self.ran = TrueRANStyleAttention(in_channels=2048)
        
        # 3. Final Classification Head
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(2048, num_classes)

    def forward(self, x):
        # Pass through the Deep PCSA-injected ResNet
        x = self.trunk(x)
        
        # Final global/local refinement with RAN
        x = self.ran(x)
        
        # Pool and Classify
        out = self.pool(x)
        out = torch.flatten(out, 1)
        out = self.fc(out)
        
        return out

# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DeepPCSARANHybrid(num_classes=102).to(device)
print("Deep PCSA + RAN Hybrid Model is ready for training!")

Building Deep PCSA ResNet-50 Trunk...
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 188MB/s]


Attaching RAN Module...
Deep PCSA + RAN Hybrid Model is ready for training!


In [4]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

DATASET_TRAIN_PATH = '/kaggle/working/ip102_organized/train'
DATASET_VAL_PATH = '/kaggle/working/ip102_organized/val'   

# CRITICAL: Decrease batch size so the GPU doesn't run out of memory with the larger images
BATCH_SIZE = 32 

# 1. Change Resize to 224x224 (ImageNet standard)
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Reload datasets
train_dataset = datasets.ImageFolder(root=DATASET_TRAIN_PATH, transform=transform_train)
val_dataset = datasets.ImageFolder(root=DATASET_VAL_PATH, transform=transform_val)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Loaded {len(train_dataset)} training images at 224x224 resolution.")

Loaded 45095 training images at 224x224 resolution.


In [7]:
import torch.optim as optim
import torch.nn as nn

criterion = nn.CrossEntropyLoss()

# Separate the parameters into two groups
new_params = []
base_params = []

# Update the filter to catch PCSA, RAN, and the Final Classifier
for name, param in model.named_parameters():
    if 'pcsa' in name or 'ran' in name or 'fc' in name:
        new_params.append(param)
    else:
        base_params.append(param)

# Assign different learning rates 
optimizer = optim.Adam([
    {'params': base_params, 'lr': 1e-5}, # Slow: Protects the ResNet trunk
    {'params': new_params, 'lr': 1e-4}   # Fast: Warms up PCSA, RAN, and FC
])

# Excellent choice of scheduler for breaking plateaus!
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=2)

print("Differential Optimizer and Scheduler Ready!")

Differential Optimizer and Scheduler Ready!


In [ ]:
import torch
import os
from tqdm import tqdm

# --- 7. TRAINING LOOP (WITH DYNAMIC SCHEDULER & CHECKPOINTING) ---
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs, device):
    # Lists to store metrics for plotting
    train_losses = []
    train_accs = []
    val_losses = []
    val_accs = []
    
    # Track the best validation accuracy for checkpointing
    best_acc = 0.0 

    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        # --- Training Phase ---
        model.train() # Ensure model is in training mode
        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in tqdm(train_loader, desc=f'Training Epoch {epoch+1}'):
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        # NOTE: scheduler.step() was REMOVED from here!

        epoch_train_loss = running_loss / len(train_loader.dataset)
        epoch_train_acc = running_corrects.double() / len(train_loader.dataset)
        train_losses.append(epoch_train_loss)
        train_accs.append(epoch_train_acc.item()) # .item() to get scalar from tensor

        print(f'Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f}')

        # --- Validation Phase ---
        model.eval() # Set model to evaluation mode
        val_running_loss = 0.0
        val_running_corrects = 0

        with torch.no_grad(): # Disable gradient calculation for validation
            for inputs, labels in tqdm(val_loader, desc=f'Validation Epoch {epoch+1}'):
                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                val_running_loss += loss.item() * inputs.size(0)
                val_running_corrects += torch.sum(preds == labels.data)
        
        epoch_val_loss = val_running_loss / len(val_loader.dataset)
        epoch_val_acc = val_running_corrects.double() / len(val_loader.dataset)
        val_losses.append(epoch_val_loss)
        val_accs.append(epoch_val_acc.item())

        print(f'Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}')

        # 1. NEW SCHEDULER PLACEMENT: ReduceLROnPlateau needs to see the validation accuracy
        scheduler.step(epoch_val_acc)
        
        # 2. CHECKPOINT SAVING LOGIC
        if epoch_val_acc > best_acc:
            best_acc = epoch_val_acc
            print(f"🌟 New best accuracy ({best_acc:.4f})! Saving checkpoint...")
            
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_acc': best_acc,
                'train_losses': train_losses,
                'train_accs': train_accs,
                'val_losses': val_losses,
                'val_accs': val_accs
            }
            # Save the checkpoint safely to Google Drive
            torch.save(checkpoint, '/kaggle/working/ip102_deep_hybrid_checkpoint.pth')

    print('\nTraining complete!')
    return model, train_losses, train_accs, val_losses, val_accs

# --- EXECUTE THE TRAINING ---
print("Starting training...")
trained_model, train_losses, train_accs, val_losses, val_accs = train_model(
    model, train_loader, val_loader, criterion, optimizer, scheduler, 10, device
)

# --- 3. FINAL MODEL SAVING LOGIC ---
# Once all epochs are finished, extract the brain and save it permanently for deployment
final_model_path = '/kaggle/working/ip102_deep_hybrid_FINAL.pth'
torch.save(trained_model.state_dict(), final_model_path)
print(f"🎉 Final model permanently saved to {final_model_path}!")

Starting training...

Epoch 1/10
----------


Training Epoch 1: 100%|██████████| 1410/1410 [11:38<00:00,  2.02it/s]


Train Loss: 2.1235 Acc: 0.4844


Validation Epoch 1: 100%|██████████| 235/235 [00:40<00:00,  5.83it/s]


Val Loss: 1.5396 Acc: 0.5946
🌟 New best accuracy (0.5946)! Saving checkpoint...

Epoch 2/10
----------


Training Epoch 2: 100%|██████████| 1410/1410 [11:40<00:00,  2.01it/s]


... (24 outputs left out)

Train Loss: 0.2756 Acc: 0.9142


Validation Epoch 8: 100%|██████████| 235/235 [00:40<00:00,  5.87it/s]


Val Loss: 1.5372 Acc: 0.6781

Epoch 9/10
----------


Training Epoch 9:  96%|█████████▌| 1348/1410 [11:10<00:30,  2.02it/s]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Plotting Loss Curves
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plotting Accuracy Curves
plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Training Accuracy')
plt.plot(val_accs, label='Validation Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Get all predictions and true labels for the validation set
all_preds = []
all_labels = []

trained_model.eval()
with torch.no_grad():
    for inputs, labels in tqdm(val_loader, desc='Generating Confusion Matrix'):
        inputs = inputs.to(device)
        outputs = trained_model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Compute the confusion matrix
cm = confusion_matrix(all_labels, all_preds)

# Plot the confusion matrix
plt.figure(figsize=(15, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=val_dataset.classes, yticklabels=val_dataset.classes)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix on Validation Set')
plt.show()